In [60]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [61]:
#merge all relevant dataset for the analysis
# form 04. clustering
# from 06. opten variables #make sure the codes are read in as str to not lose data
# from 07. city characteristics
# from 08. socio-econ and work change variables -- average workday file

clustering_path = '../output/data/48hr_averages/open_avg48hrs_clustered.pkl'
opten_path = '../data/data_gen/opten_hex_level.pkl'
city_char_path = '../output/data/city_char.csv'
population_density_path = '../output/data/population_density_hour12.pkl' #population_density_workday_avg.pkl

clustering_df = pd.read_pickle(clustering_path)
opten_df = pd.read_pickle(opten_path)
city_char_df = pd.read_csv(city_char_path, dtype={'h3_id': str})
population_density_df = pd.read_pickle(population_density_path)

In [62]:
# rename h3 ids to raster_id

clustering_df['raster_id'] = clustering_df['raster_id'].astype(str)

opten_df['h3_10'] = opten_df['h3_10'].astype(str)
opten_df = opten_df.rename(columns={'h3_10': 'raster_id'})

city_char_df['h3_id'] = city_char_df['h3_id'].astype(str)
city_char_df = city_char_df.rename(columns={'h3_id': 'raster_id'})

population_density_df['raster_id'] = population_density_df['raster_id'].astype(str)

In [63]:
#drop unnecessary cols
opten_cols = opten_df[['raster_id', 'letszam_besz18', 'arbev_2018', 'brutto_hozzaadott_ertek_2018', 'n_unique_teaor_kod',
       'n_unique_teaor_2', 'n_unique_teaor_betu', 'n_companies',
       'productivity', 'teaor_2_dominant', 'entropy', 'entropy_2dig',
       'entropy_letter', 'industry_A', 'industry_B', 'industry_C',
       'industry_D', 'industry_E', 'industry_F', 'industry_G', 'industry_H',
       'industry_I', 'industry_J', 'industry_K', 'industry_L', 'industry_M',
       'industry_N', 'industry_O', 'industry_P', 'industry_Q', 'industry_R',
       'industry_S']]

city_cols = city_char_df[['raster_id', 'distance_to_deak', 'shortest_path_drive']]

In [64]:
# There are 48 rows per raster_id (one per hour_reordered),  collapse to one
cluster_cols = clustering_df[['raster_id', 'work_cluster', 'work_cluster_label']].drop_duplicates()

In [65]:
# merge dfs
merged = population_density_df.copy()

sources = [
    (cluster_cols),
    (opten_cols),
    (city_cols),
]

for df in sources:
    base_ids = set(merged['raster_id'])
    matched_ids = base_ids & set(df['raster_id'])
    merged = merged.merge(df, on='raster_id', how='left')

In [66]:
os.makedirs('../output/data', exist_ok=True)

merged.to_pickle('../output/data/merged_analysis_data.pkl')
merged.to_csv('../output/data/merged_analysis_data.csv', index=False)

# Normalise data

In [67]:
#work cluster is NA when there was no working activity in the location, however there is busienss - I assign work cluster nr 3 to them - a new cluster
has_business = merged['n_companies'].notna() & (merged['n_companies'] > 0)
merged.loc[merged['work_cluster'].isna() & has_business, 'work_cluster'] = 3

print(f"{merged['work_cluster'].isna().sum()} raster_ids have companies but no cluster assigned")

# change NAs in these columns to 0 -- will just mean that there are 0 people working in those location
columns_to_update = ['work_abs_diff', 'loc_work_gaussian_opening']
merged[columns_to_update] = merged[columns_to_update].fillna(0)

15651 raster_ids have companies but no cluster assigned


In [68]:
# clusters as dummies
merged['work_cluster'] = merged['work_cluster'].astype('Int64')
merged = pd.get_dummies(merged, columns=['work_cluster'], prefix='work_cluster')

In [69]:
# Columns to exclude from standardization: the id, categorical/label columns, and all
# dummy/indicator columns (industry_*, work_cluster_*)
non_standardize_cols = ['raster_id', 'work_cluster_label', 'teaor_betu_dominant', 'teaor_2_dominant']
non_standardize_cols += [c for c in merged.columns if c.startswith('industry_')]
non_standardize_cols += [c for c in merged.columns if c.startswith('work_cluster_')]

columns_to_standardize = [c for c in merged.columns if c not in non_standardize_cols]

In [70]:
#only keep rows where there is a registered business
merged2 = merged[merged['n_companies'].notna()]

In [71]:
na_counts = merged2[columns_to_standardize].isna().sum()
na_counts = na_counts[na_counts > 0]
if len(na_counts) > 0:
    print("Columns with remaining NaN values before standardization:")
    print(na_counts)
else:
    print("No remaining NaN values in the columns to be standardized.")

Columns with remaining NaN values before standardization:
sex_fem_ratio_open              428
sex_male_ratio_open             428
arpu_low_ratio_open             133
arpu_mid_ratio_open             133
arpu_high_ratio_open            133
age_young_ratio_open            123
age_old_ratio_open              123
age_mid_ratio_open              123
sex_fem_ratio_curfew            494
sex_male_ratio_curfew           494
arpu_low_ratio_curfew           168
arpu_mid_ratio_curfew           168
arpu_high_ratio_curfew          168
age_young_ratio_curfew          162
age_old_ratio_curfew            162
age_mid_ratio_curfew            162
fem_ratio_change                572
arpu_low_change                 192
arpu_high_change                192
age_young_change                183
age_old_change                  183
age_mid_change                  183
socioecon_entr_open             108
socioecon_entr_curfew           146
income_entr_open                133
income_entr_curfew              168
income

In [72]:
merged2 = merged2.copy()
#filling in the largest shortest path for infinity values
merged2['shortest_path_drive'] = merged2['shortest_path_drive'].replace([np.inf, -np.inf], np.nan).fillna(22266)
merged2 = merged2.fillna(0)

In [73]:
# Standardize all numeric columns
scaler = StandardScaler()
merged_standardized = merged2.copy()
merged_standardized[columns_to_standardize] = scaler.fit_transform(merged2[columns_to_standardize])

In [74]:
# save data as standardised
merged_standardized.to_pickle('../output/data/merged_analysis_data_standardized.pkl')
merged_standardized.to_csv('../output/data/merged_analysis_data_standardized.csv', index=False)